In [1]:
import os

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
import bs4
import requests
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from dotenv import load_dotenv, dotenv_values

load_dotenv(override=True)
env_dict = dotenv_values(".env")
print("--- Contents of .env file ---")
for key, value in env_dict.items():
    # Masking the value so you don't accidentally print your full API key to your screen!
    masked_value = f"{value[:5]}...{value[-4:]}" if len(value) > 10 else value
    print(f"{key}: {masked_value}")

--- Contents of .env file ---
OPENAI_API_KEY: sk-pr...-PAA
GOOGLE_API_KEY: AQ.Ab...dTQA
TAVILY_API_KEY: tvly-...keLk
USER_AGENT: rag-f.../1.0
LANGSMITH_TRACING: true
LANGSMITH_ENDPOINT: https....com
LANGSMITH_API_KEY: lsv2_...2276
LANGSMITH_PROJECT: langc...demy
HF_TOKEN: hf_vG...sddc
LANGCHAIN_TRACING_V2: true
LANGCHAIN_API_KEY: lsv2_...13cd
LANGCHAIN_PROJECT: langc...demo


/var/folders/7t/_8zkzcdx12d70lrtsb5dgnrm0000gn/T/ipykernel_22957/4273366485.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [2]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
    )
),
)
blog_docs = loader.load()

#split
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=1000, chunk_overlap=50
)

#make splits
splits = text_splitter.split_documents(blog_docs)

#index
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma 

# vectorstore = Chroma.from_documents(
#     documents=splits,
#     embedding=GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001"))

hf_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
# 3. Create your Chroma vector store
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=hf_embeddings
)
retriever = vectorstore.as_retriever()



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Multi Query: Different Perspectives
template = """You are an AI language model assistant. Your task is to generate five 
different versions of the given user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines. Original question: {question}"""
prompt_perspectives = ChatPromptTemplate.from_template(template)

# 2. Build the chain (Swapped ChatOpenAI for ChatGoogleGenerativeAI)

local_llm = ChatOpenAI(
    base_url="http://127.0.0.1:1234/v1/", 
    api_key="lm-studio", 
    model="local-model", # LM Studio automatically uses whatever model is loaded in the server
    temperature=0
)

generate_queries = (
    prompt_perspectives 
    | local_llm 
    | StrOutputParser() 
    | (lambda x: x.split("\n"))
)

# 4. Run it!
questions = generate_queries.invoke({"question": "What is an LLM agent?"})
print(questions)

ValueError: Unexpected endpoint or method. (POST /chat/completions)